# Lab - Chunking: Let Recall Choose

Three tasks that cut the return policy into chunks in different ways and let the numbers choose. Claude is not involved: we test only the retrieval step, because a chunk that is never found can never be used.

| Task | What you learn |
| --- | --- |
| 1 | Fixed 400-character chunks, and recall@3 on twelve real questions |
| 2 | Chunks that follow the sections: the title stays with its rules |
| 3 | Sizes 200, 400, 800 against sections: recall and characters sent per question |

**You do not need an API key** - this lab makes no model calls at all. The ranker is BM25, a keyword score, so it runs on plain Python; production systems usually rank with embeddings or both, and the chunking question is the same either way.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Today Claude is not involved: we test only the retrieval step.
import math
import re
from collections import Counter
from pathlib import Path

POLICY = Path("policy.txt").read_text()   # the return policy: sixteen sections

WORD = re.compile(r"[a-z0-9]+")


def words(text):
    return WORD.findall(text.lower())


class BM25:
    """Keyword ranking: rare query words in a short chunk win."""

    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.docs = [Counter(words(c["text"])) for c in chunks]
        sizes = [sum(d.values()) for d in self.docs]
        self.avg = sum(sizes) / len(sizes)
        n = len(self.docs)
        df = Counter(w for d in self.docs for w in d)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        self.k1, self.b = k1, b

    def score(self, query, doc):
        size = sum(doc.values())
        norm = self.k1 * (1 - self.b + self.b * size / self.avg)
        return sum(self.idf.get(w, 0) * doc[w] * (self.k1 + 1) / (doc[w] + norm)
                   for w in set(words(query)))

    def search(self, query, k=3):
        ranked = sorted(zip(self.chunks, self.docs), key=lambda cd: -self.score(query, cd[1]))
        return [chunk for chunk, _ in ranked[:k]]


# question -> the exact fact the answer needs
QUESTIONS = [
    ("Can I return earrings?", "earrings"),
    ("How long does a card refund take?", "5 to 10 business days"),
    ("Who pays for shipping on a wrong item?", "caused by a store error"),
    ("How do I report a damaged parcel?", "within 7 days"),
    ("Can I exchange for another size?", "a return plus a new order"),
    ("Do I pay customs on an international return?",
     "Customs duties paid on the original delivery"),
    ("Who approves a refund of 250?", "Refunds above 100 require a human"),
    ("What happens if my return parcel is lost?", "14 days after the scan"),
    ("Can I return a gift without the buyer knowing?", "store credit in the name of the person"),
    ("My jacket broke after two months. What now?", "handled as warranty claims"),
    ("Can I return shoes without the shoe box?", "original shoe box"),
    ("How do I start a return?", "selecting the item and a reason"),
]

---

## Task 1 - Fixed-size chunks, and recall measured

In [ ]:
# ============================================================
# TASK 1 - Fixed-size chunks, and recall measured
# ============================================================
#
# WHAT TO DO
#   Write chunk_fixed(): cut the text every 400 characters,
#   wherever that falls. Write recall(): index the chunks, search
#   each question, and count it a miss when its fact is in none
#   of the top three chunks.
#
# WHY IT MATTERS
#   In a retrieval system Claude can only answer from the chunks
#   you find, and a chunk that is never found can never be used.
#   So measure retrieval on its own, on real questions with the
#   fact each answer needs - before any model is involved.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then it checks whether the fact appears in the top three
#   chunks".
#
# HOW TO DO IT
#   Three blanks:
#
#       text[i:i + size]              one slice of the text
#       size                          the step between slices
#       chunk["text"]                 where the fact must appear
# ============================================================

try:
    # BEGIN SOLUTION
    def chunk_fixed(text, size=400):
        return [{"id": f"chars {i}-{i + size}", "text": text[i:i + size]}
                for i in range(0, len(text), size)]

    def recall(chunks, k=3):
        index = BM25(chunks)
        misses = []
        for question, fact in QUESTIONS:
            found = index.search(question, k)
            if not any(fact in chunk["text"] for chunk in found):
                misses.append(question)
        return 1 - len(misses) / len(QUESTIONS), misses
    # SCAFFOLD: def chunk_fixed(text, size=400):
    # SCAFFOLD:     return [{"id": f"chars {i}-{i + size}", "text": ...}   # text[i:i + size]
    # SCAFFOLD:             for i in range(0, len(text), ...)]   # size
    # SCAFFOLD:
    # SCAFFOLD: def recall(chunks, k=3):
    # SCAFFOLD:     index = BM25(chunks)
    # SCAFFOLD:     misses = []
    # SCAFFOLD:     for question, fact in QUESTIONS:
    # SCAFFOLD:         found = index.search(question, k)
    # SCAFFOLD:         if not any(fact in ... for chunk in found):   # chunk["text"]
    # SCAFFOLD:             misses.append(question)
    # SCAFFOLD:     return 1 - len(misses) / len(QUESTIONS), misses
    # END SOLUTION: replace each ... below with the value named beside it

    fixed_chunks = chunk_fixed(POLICY)
    fixed_score, fixed_misses = recall(fixed_chunks)
except (TypeError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(f"fixed 400 chars  {len(fixed_chunks)} chunks   recall@3 = {fixed_score:.0%}")
for question in fixed_misses:
    print("    missed:", question)

check("chunk_fixed", chunk_fixed=chunk_fixed, score=fixed_score, misses=fixed_misses)

---

## Task 2 - Chunks that follow the sections

In [ ]:
# ============================================================
# TASK 2 - Chunks that follow the sections
# ============================================================
#
# WHAT TO DO
#   Write chunk_by_section(): every section, with its title,
#   becomes one chunk. Measure recall the same way and compare.
#
# WHY IT MATTERS
#   The two fixed-size misses were not facts cut in half. A
#   fixed chunk mixes the end of one section with the start of
#   the next: its words point two ways, and the section title -
#   the strongest keyword - sits in another chunk than its
#   rules. A section chunk keeps the title and its rules
#   together.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The section title, the strongest keyword, sits in a
#   different chunk from its rules".
#
# HOW TO DO IT
#   Three blanks:
#
#       "\n\n"               sections are separated by a blank line
#       lines[0].title()     the section title is the chunk's id
#       chunk_by_section(POLICY)    measure the new chunks
# ============================================================

try:
    # BEGIN SOLUTION
    def chunk_by_section(text):
        chunks = []
        for block in text.split("\n\n"):
            lines = block.strip().split("\n")
            if len(lines) >= 2:
                chunks.append({"id": lines[0].title(), "text": block.strip()})
        return chunks

    section_chunks = chunk_by_section(POLICY)
    # SCAFFOLD: def chunk_by_section(text):
    # SCAFFOLD:     chunks = []
    # SCAFFOLD:     for block in text.split(...):   # "\n\n"
    # SCAFFOLD:         lines = block.strip().split("\n")
    # SCAFFOLD:         if len(lines) >= 2:
    # SCAFFOLD:             chunks.append({"id": ..., "text": block.strip()})   # lines[0].title()
    # SCAFFOLD:     return chunks
    # SCAFFOLD:
    # SCAFFOLD: section_chunks = ...   # chunk_by_section(POLICY)
    # END SOLUTION: replace each ... below with the value named beside it
    if any(c["id"] is ... for c in section_chunks):   # provided: a blank, said plainly
        raise TypeError("ellipsis")
    section_score, section_misses = recall(section_chunks)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
print(f"fixed 400 chars  {len(fixed_chunks)} chunks   recall@3 = {fixed_score:.0%}")
print(f"by section       {len(section_chunks)} chunks   recall@3 = {section_score:.0%}")
print("first ids:", [c["id"] for c in section_chunks[:4]])

check("chunk_section", chunks=section_chunks, score=section_score, fixed_score=fixed_score)

---

## Task 3 - Smaller is not more precise; bigger is not free

In [ ]:
# ============================================================
# TASK 3 - Smaller is not more precise; bigger is not free
# ============================================================
#
# WHAT TO DO
#   Measure fixed chunks of 200, 400 and 800 characters next to
#   the section chunks: recall@3, and how many characters the
#   three found chunks send to Claude per question. Pick the
#   strategy with the best recall, then the fewest characters.
#
# WHY IT MATTERS
#   Smaller chunks are more precise only when they follow the
#   structure of the content. Bigger chunks find more, but every
#   character found is sent to the model on every question.
#   Recall and size together decide - measured, not guessed.
#
# WHERE TO SEE IT IN THE LECTURE
#   "A cut in the middle of a topic creates two half-chunks that
#   both rank badly".
#
# HOW TO DO IT
#   Three blanks:
#
#       sum(len(c["text"]) for c in chunks) / len(chunks)   the average chunk
#       3 * average             three chunks are sent per question
#       (-r["recall"], r["chars_per_question"])   best recall, then smallest
# ============================================================

try:
    candidates = {f"fixed {size}": chunk_fixed(POLICY, size) for size in (200, 400, 800)}
    candidates["by section"] = section_chunks
    # BEGIN SOLUTION
    rows = []
    for name, chunks in candidates.items():
        score, _ = recall(chunks)
        average = sum(len(c["text"]) for c in chunks) / len(chunks)
        rows.append({"strategy": name, "chunks": len(chunks), "recall": round(score, 2),
                     "chars_per_question": round(3 * average)})
    best = min(rows, key=lambda r: (-r["recall"], r["chars_per_question"]))["strategy"]
    # SCAFFOLD: rows = []
    # SCAFFOLD: for name, chunks in candidates.items():
    # SCAFFOLD:     score, _ = recall(chunks)
    # SCAFFOLD:     average = ...   # sum(len(c["text"]) for c in chunks) / len(chunks)
    # SCAFFOLD:     rows.append({"strategy": name, "chunks": len(chunks), "recall": round(score, 2),
    # SCAFFOLD:                  "chars_per_question": round(...)})   # 3 * average
    # SCAFFOLD: best = min(rows, key=lambda r: ...)["strategy"]   # (-r["recall"], r["chars_per_question"])
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for r in rows:
    print(f"{r['strategy']:11} {r['chunks']:3} chunks  recall@3 {r['recall']:.0%}"
          f"  {r['chars_per_question']:5} chars sent per question")
print("best:", best)

check("chunk_tradeoff", rows=rows, best=best)

---

## Done

Chunk along the structure of your documents - sections for policies, functions for code, messages for tickets - and measure recall on real questions, with the size it sends, before you change the model or the prompt.